# Study 947 — The Buffer Ladder — the teardown

The excess-of-cash Sharpe race, the Newey-West gap tests, paired block-bootstrap CIs, the beta decomposition, the entry-point-luck dispersion, an era cut, a cost sweep, a declared-proxy fee sweep, and the live synthetic control.

Every real number is frozen from `docs/results.md` (Fingerprint `489cd6cd95e2`), 2020-08-11 → 2026-06-30. Daily **total-return** closes throughout, TR vs TR. **One execution lag:** every estimated weight — the basket's rebalance target and every beta — is formed on data through the close of day *t* and applied at *t+1*.

In [1]:
R = {'start': '2020-08-11', 'end': '2026-06-30', 'n_days': 1478, 'n_race': 1477, 'n_matched': 1225, 'matched_start': '2021-08-12', 'fp': '489cd6cd95e2', 'bufr': (7.93, 10.21, 0.777, 2.06), 'pjan': (6.61, 8.62, 0.767, 2.08), 'papr': (5.32, 7.75, 0.687, 1.82), 'pjul': (7.21, 8.2, 0.879, 2.4), 'poct': (7.33, 7.77, 0.943, 2.65), 'diy': (6.61, 7.74, 0.853, 2.3), 'diy_matched': (7.34, 11.12, 0.661, 1.59), 'mix_bufr': (6.09, 9.8, 0.621, 1.5), 'spy': (13.73, 16.92, 0.812, 2.17), 'dd_bufr': -13.73, 'dd_diy': -10.82, 'dd_pjan': -11.93, 'dd_papr': -11.87, 'dd_pjul': -10.69, 'dd_poct': -10.22, 'dd_spy': -24.5, 'gap_diy': (1.33, 1.18, 3.52, -0.076), 'gap_pjan': (1.32, 1.26, 3.78, 0.01), 'gap_papr': (2.61, 1.74, 4.7, 0.09), 'gap_pjul': (0.73, 0.55, 4.01, -0.102), 'gap_poct': (0.61, 0.4, 4.37, -0.166), 'gap_matched': (-0.96, -1.69, 2.35, -0.057), 'gap_mix': (0.3, 0.31, 2.66, -0.018), 'gap_diy_mix': (1.26, 1.39, 2.29, 0.043), 'beta_bufr': 0.579, 'beta_diy': 0.439, 'corr_bufr_diy': 0.96, 'boot_diy': (1.33, -0.81, 3.22, 10.2), 'boot_matched': (-0.96, -1.91, -0.05, 98.4), 'boot_sharpe': (-0.076, -0.209, 0.034, 91.2), 'boot_block_sens': ((5, -2.2, 0.26, 0, 3), (10, -2.0, 0.06, 0, 3), (21, -1.87, -0.06, 3, 3), (42, -1.81, -0.13, 3, 3), (63, -1.78, -0.14, 3, 3)), 'spread_mean': 4.53, 'spread_median': 4.51, 'spread_max': 8.7, 'sd_single_mean': 6.77, 'sd_basket': 6.61, 'var_reduction': 2.4, 'pair_corr': 0.889, 'var_reduction_daily': 4.2, 'var_reduction_closed_form': 4.2, 'era_e': (726, 0.639, 0.768, 1.08, 0.54, -1.22, -1.18), 'era_l': (751, 0.961, 0.945, 1.57, 1.36, -0.79, -1.14), 'cost0': (1.32, 1.18, -0.96, -1.69), 'cost25': (1.33, 1.19, -0.95, -1.66), 'fee00': (1.33, 1.18, -0.96, -1.69), 'fee20': (1.53, 1.36, -0.76, -1.34), 'fee40': (1.73, 1.54, -0.56, -0.98), 'fee_single_pct': 0.79, 'fee_extra_pct': 0.2, 'years': {2021: (11.88, 8.8, 7.51, 7.2, 9.45, 8.25, 2.26, 28.73), 2022: (-7.57, -5.29, -4.29, -2.08, -1.25, -3.2, 4.04, -18.18), 2023: (19.63, 18.18, 16.45, 19.87, 20.12, 18.66, 3.67, 26.18), 2024: (14.68, 13.45, 12.28, 13.76, 9.55, 12.26, 4.21, 24.89), 2025: (12.44, 11.29, 6.58, 12.78, 10.99, 10.4, 6.19, 17.72)}, 'syn_planted': (3.8, 4.16, 0.36, 4.84), 'syn_null_fee': (-0.2, 0.16, 0.36, 0.18), 'syn_null_clean': (0.0, 0.36, 0.36, 0.42), 'syn_null_mean': 0.11, 'syn_null_sd': 0.87, 'syn_null_maxt': 1.69, 'syn_null_fires': 0}

## 1. Two windows, never mixed

Arms needing no estimated weight (wrapper, vintages, DIY basket) race over **n = 1,477**. The beta-matched arms burn the first 252 days on an expanding, one-day-lagged OLS beta and race over **n = 1,225** (from 2021-08-12). Reported separately throughout.

## 2. The arms, excess-of-cash (BIL total return subtracted)

In [2]:
rows = [('BUFR (ladder)', R['bufr'], R['n_race']),
        ('PJAN', R['pjan'], R['n_race']),
        ('PAPR', R['papr'], R['n_race']),
        ('PJUL', R['pjul'], R['n_race']),
        ('POCT', R['poct'], R['n_race']),
        ('DIY basket', R['diy'], R['n_race']),
        ('DIY beta-matched', R['diy_matched'], R['n_matched']),
        ('SPY/BIL mix @BUFR beta', R['mix_bufr'], R['n_matched']),
        ('SPY', R['spy'], R['n_race'])]
print('%-24s %6s %8s %7s %9s %7s' % ('arm', 'n', 'ann %', 'vol %', 'exSharpe', 'HAC t'))
for name, v, n in rows:
    print('%-24s %6d %+8.2f %7.2f %+9.3f %+7.2f' % (name, n, v[0], v[1], v[2], v[3]))

arm                           n    ann %   vol %  exSharpe   HAC t
BUFR (ladder)              1477    +7.93   10.21    +0.777   +2.06
PJAN                       1477    +6.61    8.62    +0.767   +2.08
PAPR                       1477    +5.32    7.75    +0.687   +1.82
PJUL                       1477    +7.21    8.20    +0.879   +2.40
POCT                       1477    +7.33    7.77    +0.943   +2.65
DIY basket                 1477    +6.61    7.74    +0.853   +2.30
DIY beta-matched           1225    +7.34   11.12    +0.661   +1.59
SPY/BIL mix @BUFR beta     1225    +6.09    9.80    +0.621   +1.50
SPY                        1477   +13.73   16.92    +0.812   +2.17


**Read the vol column before the return column.** The wrapper returns most of the buffer arms and has the highest vol of them, so its excess Sharpe (+0.777) sits *below* the DIY basket's (+0.853) and below POCT held alone (+0.943). The risk-adjusted ranking inverts the raw-return ranking.

## 3. The gaps — wrapper minus each DIY alternative

HAC *t* on the daily return difference (Jobson-Korkie in Newey-West form; the cash leg cancels because both arms are excess-of-cash).

In [3]:
print('%-28s %6s %10s %8s %8s %9s' % ('comparison','n','gap pp/yr','HAC t','TE %','dSharpe'))
for name, key, n in [('vs DIY basket','gap_diy',R['n_race']),
                     ('vs PJAN','gap_pjan',R['n_race']),
                     ('vs PAPR','gap_papr',R['n_race']),
                     ('vs PJUL','gap_pjul',R['n_race']),
                     ('vs POCT','gap_poct',R['n_race']),
                     ('vs beta-matched DIY','gap_matched',R['n_matched']),
                     ('vs beta-matched SPY/BIL','gap_mix',R['n_matched']),
                     ('DIY vs beta-matched mix','gap_diy_mix',R['n_matched'])]:
    g = R[key]
    print('%-28s %6d %+10.2f %+8.2f %8.2f %+9.3f' % (name, n, g[0], g[1], g[2], g[3]))
print()
print('|t| >= 2 anywhere in this table? %s'
      % any(abs(R[k][1]) >= 2 for k in ('gap_diy','gap_pjan','gap_papr','gap_pjul',
                                        'gap_poct','gap_matched','gap_mix','gap_diy_mix')))

comparison                        n  gap pp/yr    HAC t     TE %   dSharpe
vs DIY basket                  1477      +1.33    +1.18     3.52    -0.076
vs PJAN                        1477      +1.32    +1.26     3.78    +0.010
vs PAPR                        1477      +2.61    +1.74     4.70    +0.090
vs PJUL                        1477      +0.73    +0.55     4.01    -0.102
vs POCT                        1477      +0.61    +0.40     4.37    -0.166
vs beta-matched DIY            1225      -0.96    -1.69     2.35    -0.057
vs beta-matched SPY/BIL        1225      +0.30    +0.31     2.66    -0.018
DIY vs beta-matched mix        1225      +1.26    +1.39     2.29    +0.043

|t| >= 2 anywhere in this table? False


## 4. The beta decomposition — where the +1.33 pp/yr actually comes from

Expanding-window, one-day-lagged OLS beta on SPY excess returns (identical to the full-sample in-sample estimate to three decimals, which is itself worth noting — the exposures are stable):

| | SPY-beta | Ann. excess return |
|---|--:|--:|
| BUFR | **0.579** | +7.93% |
| DIY basket | **0.439** | +6.61% |
| Difference | +0.140 | +1.32 pp |

The beta gap is 0.140; SPY's excess return over the window was +13.73%/yr; 0.140 × 13.73 ≈ +1.92 pp/yr — which is more than the entire +1.33 pp/yr gap. Hold beta constant and the wrapper is **-0.96 pp/yr** behind (*t* = -1.69), with the two series 0.960 correlated day to day.

> 💡 **In plain words** — the wrapper did not ladder better. It just held more stock, in five years when holding more stock paid.

## 5. Block bootstrap (2,000 draws, 21-day blocks, paired resampling)

Both arms are resampled on the *same* block indices, so the 0.960 correlation between them survives the resampling — the correct construction for a difference between two near-identical funds.

In [4]:
for name, key in [('gap vs DIY basket', 'boot_diy'),
                  ('gap vs beta-matched DIY', 'boot_matched'),
                  ('Sharpe gap vs DIY basket', 'boot_sharpe')]:
    p, lo, hi, neg = R[key]
    excl = 'EXCLUDES zero' if lo * hi > 0 else 'straddles zero'
    print('%-26s %+7.3f  95%% CI [%+7.3f, %+7.3f]  frac<0 %5.1f%%  -> %s'
          % (name, p, lo, hi, neg, excl))

gap vs DIY basket           +1.330  95% CI [ -0.810,  +3.220]  frac<0  10.2%  -> straddles zero
gap vs beta-matched DIY     -0.960  95% CI [ -1.910,  -0.050]  frac<0  98.4%  -> EXCLUDES zero
Sharpe gap vs DIY basket    -0.076  95% CI [ -0.209,  +0.034]  frac<0  91.2%  -> straddles zero


**One CI in this study excludes zero — so we audited it.** The beta-matched gap's bootstrap CI [-1.91, -0.05] *just* clears zero while its HAC *t* is only **-1.69**. The block length is a free parameter of the bootstrap, not a fact about the tape, so the first question is whether the exclusion survives changing it.

In [5]:
# Block-length sensitivity of the beta-matched gap's CI (frozen real-tape run,
# regenerated by examples/verify.py; 3 RNG seeds x 2,000 draws per block).
print('%-8s %22s %s' % ('block', 'mean 95% CI', 'excludes zero'))
n_excl = n_all = 0
for block, lo, hi, k, s in R['boot_block_sens']:
    print('%5dd   [%+7.2f, %+7.2f]      %d/%d seeds' % (block, lo, hi, k, s))
    n_excl += k; n_all += s
print('\nexcludes zero on %d of %d (block, seed) settings' % (n_excl, n_all))

block               mean 95% CI excludes zero
    5d   [  -2.20,   +0.26]      0/3 seeds
   10d   [  -2.00,   +0.06]      0/3 seeds
   21d   [  -1.87,   -0.06]      3/3 seeds
   42d   [  -1.81,   -0.13]      3/3 seeds
   63d   [  -1.78,   -0.14]      3/3 seeds

excludes zero on 9 of 15 (block, seed) settings


**It does not.** Flip to a 5- or 10-day block and the same gap straddles zero: the exclusion holds on 9 of 15 (block, seed) settings and fails on 6. A result that changes verdict when you change a nuisance parameter is not a result. The HAC *t* of **-1.69** — which prices the autocorrelation directly instead of resampling around it — is the honest summary, and it is what this study is stamped on: suggestive of a small fee drag, **short of |*t*| ≥ 2, and not rescued by the bootstrap**. For contrast, the synthetic panel's *planted* premium excludes zero at **every** block length — that is what a real gap looks like under the same sweep, and `tests/test_strategy.py` asserts it.

## 6. Entry-point luck, and why averaging barely touches it

| Measure | Value |
|---|--:|
| Rolling 1-yr best-minus-worst vintage spread (mean / median / max) | 4.53 / 4.51 / 8.70 pp |
| SD of rolling 1-yr returns, average single vintage | 6.77% |
| SD of rolling 1-yr returns, equal-weight basket | 6.61% |
| **Variance reduction — 1-year holding period** | **2.4%** |
| **Variance reduction — daily returns** | **4.2%** |
| Closed form √((1 + 3ρ)/4) at that ρ | **4.2%** |
| Mean pairwise daily correlation | **0.889** |

This is the load-bearing measurement in the study and it needs no inference at all. The closed form for N equally-correlated legs, sd(basket)/sd(leg) = √((1 + (N−1)ρ)/N), predicts **4.2%** at ρ = 0.889, N = 4; the daily tape delivers **4.2%**. The arithmetic is exact. On a one-year holding period the realised cut is smaller still (2.4%), because each vintage's path through its own buffer and cap does not average as cleanly as its daily noise does. Laddering delivers precisely what the correlation says it must — which is almost nothing.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from buffer_ladder import data, strategy as st

In [7]:
# SYNTHETIC — no market data. How much variance does averaging N legs remove,
# as a function of how correlated those legs are? Closed form: for N equally
# correlated legs of equal variance, sd(basket)/sd(leg) = sqrt((1 + (N-1)*rho) / N).
def variance_reduction(rho, n=4):
    return (1.0 - np.sqrt((1.0 + (n - 1) * rho) / n)) * 100.0

for rho in (0.0, 0.25, 0.50, 0.75, 0.889, 0.95):
    tag = '   <-- the four Power Buffer vintages' if rho == 0.889 else ''
    print('correlation %.3f  ->  averaging 4 legs cuts sd by %5.1f%%%s'
          % (rho, variance_reduction(rho), tag))

correlation 0.000  ->  averaging 4 legs cuts sd by  50.0%
correlation 0.250  ->  averaging 4 legs cuts sd by  33.9%
correlation 0.500  ->  averaging 4 legs cuts sd by  20.9%
correlation 0.750  ->  averaging 4 legs cuts sd by   9.9%
correlation 0.889  ->  averaging 4 legs cuts sd by   4.3%   <-- the four Power Buffer vintages
correlation 0.950  ->  averaging 4 legs cuts sd by   1.9%


## 7. Era cut (split 2023-07-01)

| Era | n | exSharpe wrapper / DIY | vs DIY basket | vs beta-matched DIY |
|---|--:|--:|--:|--:|
| 2020-08 → 2023-06 | 726 | +0.639 / +0.768 | +1.08 pp (*t* = +0.54) | **-1.22 pp** (*t* = -1.18) |
| 2023-07 → 2026-06 | 751 | +0.961 / +0.945 | +1.57 pp (*t* = +1.36) | **-0.79 pp** (*t* = -1.14) |

The beta-matched shortfall is negative in both halves and significant in neither. Nothing flips sign; nothing crosses the bar. Note the second era covers the high-short-rate regime, where the excess-of-cash framing bites hardest — and the conclusion does not move.

## 8. Cost sweep — and which way it cuts

| One-way cost | vs DIY basket | vs beta-matched DIY |
|---|--:|--:|
| 0 bps (gross) | +1.32 pp (*t* = +1.18) | -0.96 pp (*t* = -1.69) |
| 25 bps | +1.33 pp (*t* = +1.19) | -0.95 pp (*t* = -1.66) |

Friction is charged one-way × NAV on the **DIY** arms only — the wrapper's own trading is already inside its NAV. So a higher cost can only *flatter* the wrapper, and even at a punitive 25 bps it still fails to beat a beta-matched DIY ladder. Rebalancing the basket never / annually / quarterly / monthly moves its excess Sharpe by less than 0.001 in all four cases — four legs correlated 0.889 barely drift apart. No short leg is required on the real tape, so no borrow is charged; the machinery charges it if the matched weight ever goes negative.

## 9. The declared PROXY — the fee layer, and its sweep

A single Power Buffer vintage quotes **0.79%/yr**; the laddered wrapper adds a management fee on top of those acquired-fund fees, an incremental layer we **assume** at **0.20%/yr**. This is a quoted number, not a tape measurement — published NAV returns are already net of whatever was actually charged. It is used only to build a 'had the layer been waived' counterfactual, and it is swept:

| Extra layer waived | vs DIY basket | vs beta-matched DIY |
|---|--:|--:|
| +0.00%/yr | +1.33 pp (*t* = +1.18) | -0.96 pp (*t* = -1.69) |
| +0.20%/yr (our assumption) | +1.53 pp (*t* = +1.36) | -0.76 pp (*t* = -1.34) |
| +0.40%/yr (generous upper bound) | +1.73 pp (*t* = +1.54) | -0.56 pp (*t* = -0.98) |

The assumed layer accounts for a fifth to a half of the beta-matched shortfall, and no value in the swept range turns the wrapper into a winner. The verdict does not rest on the guess.

## 10. Synthetic control — the detector is unbiased (offline)

It never supports the real-tape stamp; it only proves the null is a fact about the tape rather than a broken harness. The panel plants a laddering premium on top of an equal-weight vintage basket, net of a planted fee, with realistic wrapper tracking noise.

In [8]:
# SYNTHETIC control — the machinery proof. Never supports the real-tape stamp.
for tag, ss, fee in [('planted premium', 1.0, 0.002),
                     ('null, fee only ', 0.0, 0.002),
                     ('null, no fee   ', 0.0, 0.000)]:
    px, truth = data.synthetic_panel(signal_strength=ss, extra_fee_ann=fee, seed=947)
    d = st.synthetic_detect(px, truth)
    print('%s : planted %+5.2f pp/yr -> recovered %+5.2f (error %+.2f), HAC t %+.2f'
          % (tag, d['expected_gap_pp'], d['gap_ann_pp'], d['error_pp'], d['t_hac']))

nulls = [st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0,
                                                   extra_fee_ann=0.0, seed=947 + s))
         for s in range(8)]
ts = np.array([n['t_hac'] for n in nulls])
gaps = np.array([n['gap_ann_pp'] for n in nulls])
print('\nnull across 8 seeds: mean gap %+.2f pp/yr (sd %.2f), max |t| %.2f, fires on %d/8'
      % (gaps.mean(), gaps.std(ddof=1), np.abs(ts).max(), int((np.abs(ts) >= 2).sum())))

planted premium : planted +3.80 pp/yr -> recovered +4.16 (error +0.36), HAC t +4.84


null, fee only  : planted -0.20 pp/yr -> recovered +0.16 (error +0.36), HAC t +0.18


null, no fee    : planted +0.00 pp/yr -> recovered +0.36 (error +0.36), HAC t +0.42



null across 8 seeds: mean gap +0.11 pp/yr (sd 0.87), max |t| 1.69, fires on 0/8


| Panel | Planted | Recovered | Error | HAC *t* |
|---|--:|--:|--:|--:|
| Planted premium (4%/yr less a 0.20%/yr fee) | +3.80 | **+4.16** | +0.36 | **+4.84** |
| Null, fee only | -0.20 | +0.16 | +0.36 | +0.18 |
| Clean null | +0.00 | +0.36 | +0.36 | +0.42 |

Across 8 null seeds: mean gap +0.11 pp/yr (sd 0.87), max |*t*| 1.69, fires on **0/8**.

## Verdict

- **Signal — None.** No gap clears |*t*| = 2 in either direction: +1.33 pp/yr vs the DIY basket (*t* = +1.18, bootstrap CI [-0.81, +3.22]) and -0.96 pp/yr vs a beta-matched DIY ladder (*t* = -1.69). The wrapper's headline outperformance is beta (0.579 vs 0.439), and risk-adjusted it is *behind* the basket it wraps (+0.777 vs +0.853) with a 2.9 pp deeper drawdown. The entry-point luck laddering exists to remove is worth a 2.4% variance reduction at ρ = 0.889. **Survivorship:** the surviving flagships of a category that has shuttered products, and an **n-of-1** wrapper over 5.9 years with one down-year.
- **Tradability — Mirage.** Nothing to bank. Long the wrapper for laddering pays a fee layer for a 2.4% variance cut plus a beta tilt an index fund sells cheaper; short the wrapper against a beta-matched DIY ladder targets -0.96 pp/yr across 2.35% tracking error at *t* = -1.69 — a coin-flip with a borrow bill. Both arms merely **tie** the dumb beta-matched SPY/BIL mix (+0.30 pp/yr, *t* = +0.31), reproducing Study 624's result one layer up the wrapper stack.